# BUSN 20800 · The Narrative — A Story-Driven Reading

*A companion to `MIDTERM_PREP.ipynb`. Read this once, slowly. It is not a reference; it is the **story** that ties every formula in the course together.*

## How this notebook is different
- `CHEATSHEET.ipynb` — reference. What to memorize.
- `MIDTERM_PREP.ipynb` — exhaustive tutorial. Every chapter with drills.
- **This notebook** — the *why*. A single arc from \"what is Big Data\" to \"how do I attack a midterm question\". Each section teaches you to **read a formula semantically** — what each symbol represents, what the operation does, and why the formula exists.

After reading this you will be able to look at any equation in the course and say out loud: *\"This says $X$. It exists because $Y$. On the exam it becomes $Z$.\"*

---

# §0 · Prologue — Why \"Big Data\" needs its own course

## The crisis of classical statistics

For a century, statistics was the science of **small $n$, small $p$**. You collected a careful sample of $n = 30$ farm plots, measured $p = 3$ covariates per plot, ran a $t$-test, declared whether the new fertilizer was better than the old. The math was clean, the conclusions had confidence intervals, and everything was underwritten by asymptotic theorems that assumed $n \to \infty$ with $p$ held fixed.

Then the world changed.

- **Lending Club** originates millions of loans with dozens of features per borrower — $n$ in the hundreds of thousands, $p$ in the tens.
- **Nielsen** scans every barcode in every household — tens of millions of purchases, hundreds of demographic covariates per household.
- **The S&P 500** generates 450 price series × 1,300 trading days of data — a dataset where $p \approx n$.

Classical methods don't just strain under this load — they *fail*. Three things break:

1. **Overfitting.** Classical methods fit as many parameters as features. With $p \approx n$, the model has enough flexibility to memorise training noise and *none* to generalise.
2. **Inference.** The $t$-tests and confidence intervals that ruled the 20th century assume $n \gg p$ and tell you whether a coefficient is \"significantly different from zero\". When you have 450 coefficients and $n = 1000$, this machinery is useless — everything \"looks significant\" by chance.
3. **Scale of action.** Lending Club does not want a $p$-value on borrower risk. They want a **probability** and a **threshold** for automatic approval. Actionable outputs require a different vocabulary.

## The three moves this course makes

BUSN 20800 is the story of **the three intellectual responses** that resolve these crises. Every module in the course is a different chapter of the same argument:

1. **Regularization** (Module 3: `03_Model_Selection`). When the data can't pin $\beta$ down, *we* pin it down — by shrinking or zeroing out coefficients through a penalty. Mathematically: a prior on $\beta$. Philosophically: *humility*. This is lasso and ridge.
2. **Empirical model evaluation** (Module 3 again). Give up on classical inference. Replace $p$-values with **held-out prediction performance** — the train/test split, $K$-fold cross-validation, information criteria. This is honest and empirical.
3. **Decision theory** (Module 4: `04_classification`). Stop asking \"is this coefficient zero?\" and start asking \"what action should we take?\" Connect estimated probabilities to cost / profit matrices via an optimal threshold. This is ROC, AUC, and $t^\star$.

## How the course maps onto the story

| module | what's the \"crisis\"? | what's the \"resolution\"? |
|---|---|---|
| **1. Python + Viz** | we have too much data to look at by eye | `groupby`, `merge`, visualization grammar |
| **2. Linear regression** | *what is the baseline model we extend from?* | OLS, OVB, interpretation |
| **3. Model selection** | classical stats overfits when $p \approx n$ | lasso, CV, AIC/BIC, stepwise |
| **4. Classification** | binary / multiclass outcomes, and how to act on them | logistic, KNN, ROC, decision thresholds |

Read this structure carefully. It is the course's thesis in four rows.

---

# §1 · How to read a formula

Every symbol in this course is a *character* in the story, and every formula is a *sentence*. If you can say out loud what a formula says in plain English, you can recover it on the exam even if you've forgotten the exact symbols.

## The cast of characters

| symbol | what it represents | how to say it in your head |
|---|---|---|
| $y_i$ | the **observed outcome** for observation $i$ | \"what actually happened for row $i$\" |
| $\mathbf x_i \in \mathbb R^p$ | the **feature vector** for observation $i$ | \"the measurements / attributes of row $i$\" |
| $\beta$ | the **parameter vector** | \"the answer to 'how does $y$ depend on $\mathbf x$?'\" |
| $\hat y_i$ | the **model's prediction** for row $i$ | \"what the model thinks should happen for row $i$\" |
| $\hat p_i$ | the **predicted probability** $P(Y_i = 1 \mid \mathbf x_i)$ | \"the model's confidence that row $i$ is a positive\" |
| $\varepsilon_i$ | the **error term** | \"the part of $y_i$ the model can't explain\" |
| $\hat\beta$ | the **estimate of $\beta$** — the single \"best\" guess | \"our answer, given the data we saw\" |
| $p(y\mid \mathbf x, \beta)$ | the **conditional density / mass** of $y$ given $\mathbf x$ and $\beta$ | \"the model's story about how $y$ arises\" |
| $L(\beta)$, $\ell(\beta)$ | **likelihood** and **log-likelihood** | \"how plausible is this $\beta$, given what we saw?\" |
| $\text{Dev}(\beta)$ | **deviance** = $-2\ell$ | the same thing, but sign-flipped so we minimize |
| $\lambda$ | **regularization strength** | \"how much do I distrust big coefficients?\" |
| $t$ | **classification threshold** | \"how confident does the model need to be before I act?\" |
| $K$ | **number of folds** (CV) or **number of neighbours** (KNN) | depends on context — always the same letter, different roles |

## The operations

- $\prod_i$ — \"for every row, multiply\". Comes from independence of observations.
- $\sum_i$ — \"for every row, add\". Comes from taking $\log$ of a product.
- $\arg\max_\beta f(\beta)$ — \"the value of $\beta$ that makes $f$ as big as possible\".
- $\arg\min_\beta f(\beta)$ — \"the value of $\beta$ that makes $f$ as small as possible\". When $f = -g$, these are equal to $\arg\max g$.
- $\partial / \partial \beta$ — \"direction of steepest increase\". Setting to zero finds optima.
- $\hat{\cdot}$ — \"this is an **estimate**, not the truth\".
- $\mathbb E[\cdot]$ — \"the average, across many hypothetical datasets\".
- $\propto$ — \"proportional to; differs by a constant\". In $\arg\max$ contexts, proportionality and equality are interchangeable.

## How to read a sentence

Take a typical formula:
$$\hat\beta = \arg\min_\beta \sum_{i=1}^n (y_i - \mathbf x_i'\beta)^2 + \lambda \sum_{j=1}^p |\beta_j|.$$

Read it right-to-left, inside-out:

1. $\mathbf x_i' \beta$ — \"the model's prediction for row $i$\" (a number).
2. $y_i - \mathbf x_i'\beta$ — \"the residual for row $i$\" (how wrong).
3. $(y_i - \mathbf x_i'\beta)^2$ — \"squared residual\". Penalizes bigger errors more.
4. $\sum_i (\cdots)^2$ — \"total squared error across the whole dataset\". This is RSS.
5. $|\beta_j|$ — \"magnitude of coefficient $j$\".
6. $\sum_j |\beta_j|$ — \"total coefficient magnitude\". This is $\|\beta\|_1$.
7. $\lambda \|\beta\|_1$ — \"penalty, scaled by how much we distrust big coefficients\".
8. RSS $+ \lambda \|\beta\|_1$ — \"total cost: misfit + complexity\".
9. $\arg\min_\beta(\ldots)$ — \"pick the $\beta$ that minimises the total cost\".
10. $\hat\beta$ — the answer.

That's the **lasso**, read aloud as English. Every formula in the course can be read this way.

## The reading exercise I want you to do

Every time you see a formula in the rest of this notebook (and in `MIDTERM_PREP`), *before* reading my explanation, try to read it yourself — out loud, right-to-left, inside-out. Only then compare to the narrative.

This is how you *own* the material instead of just recognising it.

---

# §2 · Likelihood — the grammar of the entire course

## Step 1: What is a model?

A **statistical model** is a **storytelling machine**. You hand it a guess about the parameters $\beta$, and it tells you how likely each possible outcome $y$ is for each input $\mathbf x$.

Formally: a model is the conditional density $p(y\mid \mathbf x, \beta)$. This function answers:
> \"If the world worked according to parameters $\beta$, and we saw input $\mathbf x$, how likely would the various outcomes $y$ be?\"

## Step 2: What is a likelihood?

Now flip the perspective. We have **observed** data $(\mathbf x_i, y_i)_{i=1}^n$ — the world already happened, we can't change it. What we can change is our **guess about $\beta$**.

The **likelihood** $L(\beta)$ is the model's judgement of *how well the guess $\beta$ explains what we actually saw*:
$$L(\beta) \;=\; p(y_1, \ldots, y_n \mid \mathbf x_1, \ldots, \mathbf x_n, \beta) \;=\; \prod_{i=1}^n p(y_i \mid \mathbf x_i, \beta).$$

**How to read this.**
- Inside the product: \"for each row, how likely was its outcome under my proposed $\beta$?\"
- The product: \"multiplying across rows assumes independence — each row's outcome is its own die roll\".
- The whole function: \"given the data, this is a curve over all possible $\beta$ values. Its maximum is my best guess.\"

**Terminology detail.** $p(y_i \mid \mathbf x_i, \beta)$ is a *density* when $y$ is continuous and a *probability mass function* when $y$ is discrete. Both are \"how plausible is the outcome\". Don't get hung up on the distinction — for midterm purposes they behave identically.

## Step 3: Why we take logs

Products of probabilities get numerically tiny (try multiplying 1000 numbers all $< 1$). They also have ugly derivatives. So we **always** take the log:
$$\ell(\beta) = \log L(\beta) = \sum_{i=1}^n \log p(y_i \mid \mathbf x_i, \beta).$$

Two crucial facts:
1. $\log$ is strictly increasing, so $\arg\max L = \arg\max \ell$. **Optimizing the log is equivalent to optimizing the likelihood.**
2. $\log(\prod) = \sum$, turning a product of $n$ numbers into a sum of $n$ numbers — much easier derivatives.

## Step 4: Deviance — same content, different sign

For historical reasons, the **deviance** is $\text{Dev}(\beta) = -2\,\ell(\beta)$. Two facts:
1. $\arg\max \ell = \arg\min \text{Dev}$. (Same content, flipped sign.)
2. The factor of 2 is a convention from likelihood-ratio testing. For midterm purposes just remember: **deviance is minimised, likelihood is maximised, they point at the same $\hat\beta$.**

## Why this grammar is everywhere

Every parametric estimate $\hat\beta$ in this course is the solution to *some* version of this triangle:
$$\underbrace{L(\beta)}_{\text{model story quality}} \;\to\; \underbrace{\ell(\beta)}_{\text{log-story quality}} \;\to\; \underbrace{\text{Dev}(\beta) = -2\ell(\beta)}_{\text{minimisable cost}}.$$

The specific shape of $p(y\mid \mathbf x, \beta)$ changes — Gaussian for linear regression, Bernoulli for logistic — but the **mental pipeline is identical**.

### Connects to the exam
- **MC2 on Midterm Practice**: \"Which quantity is minimized in linear regression?\" Answer = RSS, because Gaussian $\text{Dev}$ collapses to RSS after dropping constants.
- **MC2 on Midterm Practice** (different version): \"Deviance is minimized / related to likelihood / applies to any GLM\" — all derived from the triangle above.
- **Short-answer on Assignment 2 Part B** and the probit version on **Midterm Practice 1.1**: every likelihood derivation starts with this triangle.

---

# §3 · Linear regression — the model as a \"what-if\" machine

## Step 1: What is linear regression saying?

The linear model
$$y_i = \beta_0 + \beta_1 x_{i1} + \cdots + \beta_p x_{ip} + \varepsilon_i$$
is a very specific storytelling machine. It says:

> *There is a true \"latent\" function of $\mathbf x$ that determines the average value of $y$. That function is a straight-line combination of the features. Every observed $y_i$ differs from this average by pure Gaussian noise $\varepsilon_i$.*

The coefficients $\beta_j$ encode the answer to a **counterfactual**:
> *If I bumped $x_j$ up by 1 unit, holding all other features fixed, by how much would the expected value of $y$ change?*

Answer: $\beta_j$. That's what the coefficient *means*. The regression output is a table of answers to \"what if\" questions.

## Step 2: How OLS emerges from the likelihood grammar

Gaussian errors give us a specific likelihood (§2 grammar instantiated):
$$p(y_i\mid \mathbf x_i, \beta) = \tfrac{1}{\sigma\sqrt{2\pi}} \exp\!\left(-\tfrac{(y_i - \mathbf x_i'\beta)^2}{2\sigma^2}\right).$$

**How to read this density:** the outcome $y_i$ is centered on the model's prediction $\mathbf x_i' \beta$ with standard deviation $\sigma$. The further $y_i$ is from the prediction, the less likely.

Multiply across rows, take logs, drop constants in $\beta$:
$$\arg\max_\beta \ell(\beta) \;=\; \arg\min_\beta \sum_i (y_i - \mathbf x_i'\beta)^2 \;=\; \arg\min_\beta \text{RSS}(\beta).$$

**OLS is nothing more than \"pick the $\beta$ that makes the Gaussian story most plausible\".** The formula $\hat\beta = (X'X)^{-1}X'\mathbf y$ is the calculus-based closed-form solution to this minimization.

### How to read $\hat\beta = (X'X)^{-1}X'\mathbf y$

Right-to-left:
1. $\mathbf y$ — the outcomes.
2. $X'\mathbf y$ — \"project $\mathbf y$ onto each feature\". Element $j$ is $\sum_i x_{ij} y_i$: the raw correlation between feature $j$ and the outcome.
3. $X'X$ — the **feature-correlation matrix**. Element $(j,k)$ is $\sum_i x_{ij} x_{ik}$: how feature $j$ and feature $k$ co-vary.
4. $(X'X)^{-1}$ — the *inverse* of that. Reading this loosely: \"when features are correlated, naive correlations of $\mathbf y$ with a single feature double-count information already in other features; the inverse subtracts out that redundancy\".
5. $(X'X)^{-1} X' \mathbf y$ — the \"purified\" correlation of each feature with $\mathbf y$, holding other features constant. That is exactly what a regression coefficient is.

In short: **$\hat\beta$ is the correlation of each feature with $\mathbf y$, *after removing the parts already explained by other features*.**

## Step 3: Reading a regression output

Imagine `statsmodels` gives you:

| | coef | std err | t |
|---|---|---|---|
| Intercept | 2.3 | 0.1 | 23 |
| `loan_amnt` | 0.0014 | 0.0003 | 4.7 |
| `dti` | 0.0007 | 0.0001 | 7 |
| `log_income` | -0.012 | 0.002 | -6 |

**What each number tells you, as a sentence:**
- $\hat\beta_0 = 2.3$: \"a hypothetical borrower with 0 loan, 0 DTI, and 1 dollar income would face an interest rate of about 2.3 (probably meaningless — no one looks like this).\"
- $\hat\beta_{\text{loan}} = 0.0014$: \"each additional \$1,000 of loan size is associated with an interest rate 0.14 percentage points higher.\"
- $\hat\beta_{\text{dti}} = 0.0007$: \"each unit higher DTI is associated with 0.07pp higher rate.\"
- $\hat\beta_{\text{log\_income}} = -0.012$: \"a 1% higher annual income is associated with a $0.012/100 = 0.00012$-point lower rate; doubling income (+log 2 ≈ +0.7) is associated with $-0.012 \cdot 0.7 \approx -0.8$pp lower rate.\"

**On the exam you will be asked to translate coefficients into sentences exactly like these.**

## Step 4: The four log / level flavours — not four formulas, four questions

Don't memorise the four-case interpretation table (§11.2 of MIDTERM_PREP) as four random rules. They're all the same single rule — *a one-unit change in \"whatever is on the right\" corresponds to a $\beta$-unit change in \"whatever is on the left\"* — applied to four different combinations of what's been transformed.

Transforming to $\log$ changes \"unit\" to \"relative change (%)\":
- $\Delta(\log x) \approx 0.01 \Leftrightarrow \Delta x/x \approx 1\%$.

So the four cases are:

| LHS | RHS | unit on LHS | unit on RHS | $\beta$ means |
|---|---|---|---|---|
| $y$ | $x$ | absolute | absolute | absolute on $y$ per absolute on $x$ |
| $\log y$ | $x$ | relative | absolute | % on $y$ per absolute on $x$ |
| $y$ | $\log x$ | absolute | relative | absolute on $y$ per % on $x$ |
| $\log y$ | $\log x$ | relative | relative | % on $y$ per % on $x$ = **elasticity** |

One rule, four wrappings.

## Step 5: OVB — when your model's story has a hole

$$E[\hat\beta_1^{\text{short}}] - \beta_1 \;=\; \beta_2 \cdot \tfrac{\text{Cov}(x_1, x_2)}{\text{Var}(x_1)}.$$

**Read this as:** \"If I forgot to include $x_2$, my estimate of $\beta_1$ is contaminated by a term that depends on (a) how much $x_2$ really affects $y$ (that's $\beta_2$) and (b) how strongly $x_2$ moves together with $x_1$ (that's the correlation). If either factor is zero, no bias.\"

The **sign rule** is the same idea stripped to sign:
$$\text{sign(bias)} = \text{sign}(\beta_2) \cdot \text{sign}(\text{corr}(x_1, x_2)).$$

### Assignment 2 Q4 worked as a sentence
- `loan_amnt` is correlated with `log_income` (bigger loans go to richer people): sign = +.
- `log_income` has a negative effect on interest rate: $\beta_{\text{income}} < 0$.
- Product of signs: (−)(+) = **−**.
- Interpretation: when I forget income, my coefficient on `loan_amnt` is **biased downward**. Adding income should make it *larger*.
- Observation: 0.00114 → 0.00148. **Larger, as predicted.** ✓

OVB is not a trick — it's a consequence of the story your model tells being incomplete.

### Connects to the exam
- **Short-answer**: \"Explain why coefficient X changes when you add variable Y\" — every answer is an OVB story.
- **Midterm Practice Concept Check 1** (logistic interpretation) uses the same grammar, shifted to log-odds.

---

# §4 · Logistic regression — when the outcome is yes / no

## Step 1: Why linear regression breaks for binary $y$

If you try to fit $y = \mathbf x' \beta + \varepsilon$ when $y \in \{0, 1\}$:
- The model can predict $\hat y < 0$ or $\hat y > 1$ — not a probability.
- The variance of $\varepsilon$ depends on $\mathbf x$ (residual is either $-\hat p$ or $1-\hat p$) — so OLS standard errors are wrong.

The fix: don't model $y$ directly; model $P(Y = 1 \mid \mathbf x)$ and squash it into $(0, 1)$ with a **link function**.

## Step 2: The GLM framework — the unifying idea

A **Generalized Linear Model** (GLM) keeps the linear combination $\mathbf x' \beta$, but passes it through a non-linear $f$:
$$P(Y_i = 1 \mid \mathbf x_i) = f(\mathbf x_i' \beta).$$

$f$ can be anything that maps $\mathbb R$ monotonically into $(0, 1)$. Two choices matter:

- **Logit** (logistic regression): $f = \sigma$, the sigmoid $\sigma(z) = 1/(1 + e^{-z})$.
- **Probit**: $f = \Phi$, the standard-normal CDF.

Both are \"S-curves\" — small $z$ maps to $\approx 0$, large $z$ maps to $\approx 1$, and the transition is smooth. Visually almost indistinguishable; mathematically they correspond to different **latent-error distributions**.

## Step 3: Reading the log-odds

For the logit link, the identity $\sigma^{-1}(p) = \log\tfrac{p}{1-p}$ (called the *logit*) gives:
$$\log\frac{P(Y=1\mid \mathbf x)}{P(Y=0\mid \mathbf x)} = \mathbf x' \beta.$$

**This is linear regression, but for the log-odds.** Every intuition from §3 about linear regression transfers — the only change is **what's on the left-hand side**.

### How to read a logistic coefficient
If $\hat\beta_j = 0.7$:
- **On the log-odds scale:** a 1-unit increase in $x_j$ increases the log-odds by 0.7. Log-odds add.
- **On the odds scale:** odds multiply by $e^{0.7} \approx 2.01$. Odds multiply.
- **On the probability scale:** depends on the baseline probability. Near $p = 0.5$, probability changes by $\approx \sigma'(0) \cdot 0.7 = 0.25 \cdot 0.7 = 0.175$. In the tails, the change is far smaller.

**The classic exam trap:** saying \"$\hat\beta_j$ is the change in probability\" is wrong. It's the change in **log-odds**. The midterm tests this in multiple-choice.

## Step 4: The score function — *what it really says*

Derive $\partial \ell / \partial \beta$ for the logistic likelihood (MIDTERM_PREP §4.4 does this in 4 lines). The answer is:
$$\frac{\partial \ell}{\partial \beta} \;=\; \sum_{i=1}^n (y_i - \hat p_i) \, \mathbf x_i.$$

**This formula, read as a sentence:**
> *\"The gradient of the log-likelihood is a sum over rows of **residuals** ($y_i - \hat p_i$) weighted by **feature vectors** ($\mathbf x_i$).\"*

Three pieces of intuition packed into this:
1. $(y_i - \hat p_i)$ is the **prediction error**. Positive if the model under-estimates, negative if it over-estimates.
2. Multiplying by $\mathbf x_i$ means observations with extreme feature values \"vote\" more on where $\beta$ should move.
3. Setting this to zero says: at the MLE, **the residuals and the features are uncorrelated**. The model has extracted everything it can from $\mathbf x$; what's left is pure noise.

This is exactly the OLS score too — there, residuals are $y_i - \hat y_i$ and features are $\mathbf x_i$. OLS and logistic MLE share the same *shape* of score function.

## Step 5: Gradient descent — why we need it

For OLS, the score $\sum (y_i - \mathbf x_i' \beta)\mathbf x_i = 0$ is **linear in $\beta$**. Solve it in closed form.

For logistic regression, $\hat p_i = \sigma(\mathbf x_i' \beta)$ is **nonlinear in $\beta$**. The score $\sum(y_i - \sigma(\mathbf x_i'\beta))\mathbf x_i = 0$ has no algebraic solution.

So we **iterate**. Start at some $\beta_0$; compute the gradient; take a small step in the gradient's direction; repeat. Stop when $\beta$ stops changing.
$$\beta_{t+1} = \beta_t - \eta \, \nabla \text{Dev}(\beta_t).$$

Assignment 2 Part B implements exactly this. The deepest insight: **gradient descent is why logistic regression exists as a practical method** — once computers could iterate, non-closed-form GLMs became tractable.

## Step 6: Probit — same story, Gaussian narrator

Probit replaces $\sigma$ with $\Phi$ (Gaussian CDF). The derivations are word-for-word identical but produce:
$$S(\beta) = \sum_i \frac{\phi(\mathbf x_i'\beta)(y_i - \Phi(\mathbf x_i'\beta))}{\Phi(\mathbf x_i'\beta)(1 - \Phi(\mathbf x_i'\beta))} \mathbf x_i.$$

**Same shape as logistic**: a weighted residual $\times$ feature vector, summed across rows. The weight is ugly because $\Phi'$ isn't a clean multiple of $\Phi(1-\Phi)$ the way $\sigma'$ is. The derivation is **Midterm Practice 1.1**.

The deeper point: both logit and probit are concrete choices within the GLM framework. **The framework is what you're really learning.**

### Connects to the exam
- **MC 3**: logistic β = log-odds.
- **Assignment 2 Part B**: derive logistic score.
- **Midterm Practice 1.1**: derive probit score.
- **Assignment 4 Q4–Q5**: multinomial logistic (same framework, $K$-way softmax instead of binary).

---

# §5 · Regularization — humility, written in math

## Step 1: The crisis revisited

When $p \approx n$ (or $p > n$), OLS overfits. The $(X'X)^{-1}$ in the closed form becomes ill-conditioned — small changes in $\mathbf y$ produce huge changes in $\hat\beta$. The fitted model tracks training noise and generalises poorly.

In **Big Data**, this is the default regime. Lending Club has $n$ in the hundreds of thousands, but once you one-hot encode borrower state, loan purpose, employment title, …, $p$ also climbs into the hundreds. The S&P 500 tracking problem had $p = 450, n = 1073$. Classical OLS produces test $R^2$ that is worse than predicting $\bar y$.

## Step 2: The idea of a penalty

If the data can't pin $\beta$ down, *we* pin it down. We tell the estimator:
> *\"You're allowed to fit the data, but you pay a price for every non-zero coefficient. Find the $\beta$ that gets the best compromise.\"*

The lasso penalizes coefficient magnitudes by their $L_1$ norm:
$$\hat\beta_{\text{lasso}} = \arg\min_\beta \underbrace{\text{RSS}(\beta)}_{\text{fit}} + \underbrace{\lambda \|\beta\|_1}_{\text{penalty}}.$$

**How to read the objective:**
- RSS — \"how badly I fit the data\".
- $\|\beta\|_1 = \sum_j |\beta_j|$ — \"how 'complex' my model is, measured by total coefficient magnitude\".
- $\lambda$ — \"how expensive complexity is, relative to misfit\".
- $\arg\min$ of the sum — \"find the $\beta$ that balances fit and complexity\".

Ridge does the same with $\|\beta\|_2^2 = \sum_j \beta_j^2$.

## Step 3: Why lasso zeros coefficients but ridge shrinks smoothly

This is the **geometric intuition** that should underlie how you think about the two methods.

- The $L_1$ ball $\{\beta : \|\beta\|_1 \le t\}$ is a **rotated square** (or cross-polytope in higher dim) with *corners* on the coordinate axes.
- The $L_2$ ball $\{\beta : \|\beta\|_2 \le t\}$ is a **smooth sphere** with no corners.

The lasso optimum is the point where the RSS ellipse first touches the feasibility region. **A convex ellipse touching a square almost always hits a corner first** — and corners are points where some coordinates are zero. So lasso produces exactly-zero coefficients. A convex ellipse touching a sphere almost never lands at an axis — so ridge's coefficients shrink smoothly toward zero but don't hit it exactly.

**That's the one-line story for why lasso selects features and ridge doesn't.**

## Step 4: Bayesian reinterpretation — the prior as belief

Here's the move that unifies regularization with everything else:
$$\text{Penalty in the objective} \;\Longleftrightarrow\; \text{Prior belief about }\beta.$$

Recall from §2: the MAP estimate maximises $L(\beta) \cdot p(\beta)$, i.e., likelihood times prior. Taking logs: $\ell(\beta) + \log p(\beta)$.

Now pick a prior $p(\beta)$ and see what $\log p(\beta)$ looks like:

- **Laplace**$(0, b)$: $\log p(\beta) = -\tfrac 1 b \sum_j |\beta_j| + c$. The $\log p$ contributes an $L_1$ penalty.
- **Gaussian**$(0, \tau^2)$: $\log p(\beta) = -\tfrac{1}{2\tau^2} \sum_j \beta_j^2 + c$. Contributes an $L_2$ penalty.

Combining with Gaussian errors (which give $\ell = -\text{RSS}/2\sigma^2 + c$):

- $\arg\max_\beta [\ell + \log p_{\text{Laplace}}]$ = $\arg\min_\beta [\text{RSS} + (2\sigma^2/b) \|\beta\|_1]$ = **lasso**.
- $\arg\max_\beta [\ell + \log p_{\text{Gaussian}}]$ = $\arg\min_\beta [\text{RSS} + (\sigma^2/\tau^2) \|\beta\|_2^2]$ = **ridge**.

$$\boxed{\text{Lasso = MAP with Laplace prior. Ridge = MAP with Gaussian prior.}}$$

### How to read the prior
- **Laplace$(0, b)$**: *\"a priori I believe most coefficients are near zero, with small chance of big outliers.\"* The sharp peak at zero is what drives sparsity. Small $b$ = strong belief in small coefficients = big $\lambda$.
- **Gaussian$(0, \tau^2)$**: *\"a priori I believe coefficients are small with no particular attraction to exactly zero.\"* The smooth curvature at zero is why ridge shrinks without zeroing out.

## Step 5: Why standardization is non-negotiable

The penalty $\lambda\|\beta\|_1$ treats \"one unit of $\beta_j$\" the same for every feature. If feature $j$ is measured in dollars and feature $k$ in thousands of dollars, the dollar feature naturally carries ~1000× larger $\beta$ — and the penalty shrinks it ~1000× more aggressively.

*This is an artifact of units, not of signal.*

Fix: before running lasso/ridge, standardize each feature to mean 0 and variance 1. Now \"one unit of $\beta_j$\" means the same thing (one standard deviation of effect size) for every feature.

### Connects to the exam
- **Assignment 3 Q1** asks for the MAP derivation — the exact content of Step 4 above.
- **Midterm Practice MC 5**: \"As $\lambda$ increases in lasso…\" — tests Step 3.
- **Midterm Practice Concept 3**: \"Why is standardization important?\" — tests Step 5.
- **Assignment 3 Q4** contrast between lasso and forward stepwise explains **shrinkage bias**: even selected lasso coefficients are biased toward zero, so forward stepwise + OLS can beat lasso on out-of-sample fit.

---

# §6 · Model selection — picking your humility

## Step 1: The hyperparameter dilemma

Regularization gave us a family of models indexed by $\lambda$:
- $\lambda = 0$: OLS (overfits with large $p$).
- $\lambda = \infty$: everything zero (underfits).
- There is some *best* $\lambda^\star$ in between.

Classical statistics has no good way to pick $\lambda^\star$. Test-set error depends on a sample you don't have. Training error is a lying signal (monotonically non-increasing in flexibility).

So Big Data uses **empirical out-of-sample evaluation**. The train/validation/test split is the simplest form; cross-validation is the generalization.

## Step 2: $K$-fold cross-validation as a rehearsal

Imagine you're preparing for the midterm. You have 5 practice tests. How do you know which study strategy works best? Take each practice test after using strategy A, then after using strategy B, then ... average the scores. The strategy with the best average is your best guess.

That's cross-validation, where:
- \"Study strategy\" = $\lambda$ value.
- \"Practice test\" = a held-out fold of the training data.
- \"Score\" = validation MSE or deviance.

Operationally, with $n$ rows and $K$ folds:
- Split data into $K$ roughly equal groups.
- For each fold $k$: train on the other $K-1$ folds ($n(K-1)/K$ rows), test on fold $k$ ($n/K$ rows).
- Average the $K$ validation losses.
- Repeat for each candidate $\lambda$; pick the one with smallest average loss.

**Exam arithmetic:** expect to be asked \"how many fits, how many rows per fit\" for given $n$ and $K$.

## Step 3: The 1-SE rule — humility about the rehearsal

The CV loss curve is **itself estimated with noise** — each of the $K$ validation losses is based on only $n/K$ points. So the minimum is only \"approximately\" the minimum; many $\lambda$'s are statistically indistinguishable.

**1-SE rule:** among all $\lambda$'s whose mean CV loss is within **one standard error** of the minimum, pick the *largest* $\lambda$. You lose negligible validation performance and get a more parsimonious model.

Philosophy: when two models are \"equally good\" up to estimation noise, pick the simpler one. This is Occam in its statistical form.

## Step 4: AIC and BIC — theoretical shortcuts

CV is great but expensive — $K$ fits per candidate $\lambda$. For huge datasets, we want a faster proxy.

**AIC:** $-2\ell + 2k$. Adds $2$ per extra parameter.
**BIC:** $-2\ell + k \log n$. Adds $\log n$ per extra parameter.

**How to read these:** both start from deviance ($-2\ell$) and *penalize complexity*. AIC has a constant penalty per parameter; BIC's penalty scales with $\log n$ and so punishes complexity more as data grows.

Key relationship: $\log n > 2$ whenever $n > e^2 \approx 7.4$. **For any realistic dataset, BIC penalizes more than AIC, so BIC picks smaller models.**

## Step 5: Why AIC and BIC fail on time-series data

Both criteria use the **raw** $n$ as the \"amount of information\". But with autocorrelated data (daily stock prices, say), consecutive observations carry much less independent information than iid draws. The *effective* sample size is smaller than $n$. The criteria think they have more data than they do, so their complexity penalties are too small, and they pick too-big models.

Assignment 3 Q5 is a live demonstration: on S&P tracking, AIC picks 313 stocks, BIC 219, while CV (which estimates generalization empirically) picks 17. The 17-stock portfolio generalizes better.

**`TimeSeriesSplit`** is the CV variant that respects time order — expanding training window, validation fold strictly after training. No leakage, realistic out-of-sample estimates.

## Step 6: Forward stepwise — greedy alternative

At each step, add the one feature that most reduces training RSS. Stop at target size.

**Crucial contrast with lasso**: forward stepwise uses **unshrunken OLS coefficients** on its selected features. Lasso's coefficients are shrunken toward zero even for selected features.

This is why in Assignment 3 Q4, forward stepwise with 30 features beat lasso with ~30 non-zero features on the S&P test set. Lasso's shrinkage bias cost it.

Modern refinement: **relaxed lasso** = use lasso for feature selection, then refit OLS on the selected subset. Best of both worlds.

### Connects to the exam
- **Review slide 13** explicitly asks: \"How many times do we need to fit our model for $K$-fold CV?\"
- **Assignment 3** is entirely about this chapter.
- **Concept-check arithmetic** on $K$-fold fit counts is near-guaranteed.

---

# §7 · Classification evaluation — probabilities into actions

## Step 1: What logistic regression gives you

Logistic regression doesn't output \"yes\" or \"no\". It outputs a probability $\hat p_i \in (0, 1)$. Somewhere between the probability and the action (\"approve this loan\", \"send this mailer\", \"offer this tutoring\") is a **decision**.

That decision is where business meets statistics. This chapter is the grammar of turning probabilities into decisions.

## Step 2: The confusion matrix — the audit

Pick a threshold $t$. Declare $\hat y_i = \mathbb 1(\hat p_i > t)$. Count four things:

|  | predicted 0 | predicted 1 |
|---|---|---|
| actually 0 | TN | FP |
| actually 1 | FN | TP |

Every classification metric is a function of these four numbers.

### How to read the matrix
- **TP (true positive)**: you said \"1\", it really was \"1\". Right call.
- **FP (false positive / Type I)**: you said \"1\", it was \"0\". Wrong alarm.
- **FN (false negative / Type II)**: you said \"0\", it was \"1\". Missed the target.
- **TN (true negative)**: you said \"0\", it really was \"0\". Quiet success.

## Step 3: Every rate asks a different question

A rate is always `numerator / denominator`. The denominator tells you what the rate is conditioned on:

| rate | formula | denominator | what question it answers |
|---|---|---|---|
| Accuracy | $(TP+TN)/n$ | everyone | \"of all predictions, what fraction were right?\" |
| **TPR = sensitivity = recall** | $TP/n_P$ | positives ($n_P = TP + FN$) | \"of the actual positives, what fraction did I catch?\" |
| **TNR = specificity** | $TN/n_N$ | negatives | \"of the actual negatives, what fraction did I correctly dismiss?\" |
| FPR | $FP/n_N = 1-$TNR | negatives | \"of the actual negatives, what fraction did I falsely flag?\" |
| FNR | $FN/n_P = 1-$TPR | positives | \"of the actual positives, what fraction did I miss?\" |
| Precision (PPV) | $TP/(TP+FP)$ | predicted positives | \"of the ones I flagged, what fraction are real?\" |

**Why so many metrics?** Because which matters depends on the business question:
- A cancer screening wants high **sensitivity** — don't miss real cancers.
- A spam filter wants high **specificity** — don't falsely mark real email.
- A marketing campaign wants high **precision** — don't waste mailers on non-buyers.

## Step 4: The base-rate trap

If only 10% of observations are positive (rare-class problem) and you use $t = 0.5$, most $\hat p_i$ are below 0.5 and the model almost always predicts 0. This gives:
- very high **specificity** (correctly saying 0 for most negatives),
- very low **sensitivity** (missing most of the rare positives),
- high **accuracy** (because 90% of \"0\" predictions happen to be right, driven by the base rate).

This is the **accuracy paradox**: a \"predict 0 always\" classifier can look great by accuracy alone. Always report both sens and spec for rare-class problems, and consider lowering the threshold.

Assignment 4 Q6 showed exactly this for Bud Light (base rate ~30%): at $t = 0.5$, sensitivity = 0.14, specificity = 0.95.

## Step 5: ROC — the menu of tradeoffs

Vary $t$ continuously from high (predict 1 for nothing) to low (predict 1 for everything). Plot (FPR, TPR) for each $t$. You get the **ROC curve**.

**How to read the ROC:**
- Bottom-left (0, 0): \"predict 1 for no one\". TPR = 0, FPR = 0. You miss every positive.
- Top-right (1, 1): \"predict 1 for everyone\". TPR = 1, FPR = 1. You catch every positive but also flag every negative.
- **Diagonal** $y = x$: random classifier. TPR = FPR at every threshold.
- **Upper-left corner** (0, 1): perfect classifier. TPR = 1 at FPR = 0.

Better classifiers \"bow\" toward the upper-left. **AUC** = area under the ROC = probability a random positive ranks above a random negative. 0.5 = chance; 1.0 = perfect.

**AUC is threshold-free.** It summarises the *ranking* quality of $\hat p$ — how well the model orders observations — independent of the specific $t$ you pick.

## Step 6: The decision-theoretic threshold

So where should $t$ live? That depends on the **cost matrix**. Set up:

|  | true $Y = 1$ | true $Y = 0$ |
|---|---|---|
| Act | $+B_{TP}$ | $-C_{FP}$ |
| Don't | $0$ | $0$ |

(For more complex problems there are costs of inaction too; the derivation generalises.)

Expected payoff of acting at $\hat p = p$:
$$\mathbb E[\text{act}\mid p] = B_{TP}\, p + (-C_{FP})(1-p).$$

Act if this exceeds 0 (the payoff of doing nothing):
$$B_{TP}\, p - C_{FP}(1-p) > 0 \;\;\Leftrightarrow\;\; p > \frac{C_{FP}}{B_{TP} + C_{FP}}.$$

$$\boxed{\;t^\star = \frac{C_{FP}}{B_{TP} + C_{FP}}.\;}$$

**How to read this formula:**
- Numerator: cost of flagging a negative (a false alarm).
- Denominator: cost of a false alarm **plus** benefit of a true positive.
- So $t^\star$ is the break-even probability — the point at which the expected gain of acting ($B_{TP} \cdot p$) exactly cancels the expected cost ($C_{FP} \cdot (1-p)$).

**When is $t^\star$ low?** When $B_{TP}$ is big relative to $C_{FP}$. You should act on weak evidence if the payoff is big and the cost is small.

**When is $t^\star$ high?** When $C_{FP}$ is big relative to $B_{TP}$. You should only act on strong evidence if false alarms are expensive.

### The universal recipe for any threshold problem
1. Write the payoff matrix. Include *both* \"act\" and \"don't act\" rows. Label both classes.
2. Compute $\mathbb E[\text{act}\mid p] - \mathbb E[\text{don't act}\mid p]$ as a function of $p$.
3. Set the difference equal to 0.
4. Solve (linearly) for $p$. Call this $t^\star$.
5. Interpret: above $t^\star$, act; below, don't.

This recipe handles *every* business decision problem in the course:
- Assignment 4 Q8 mailer: $t^\star = 0.50 / 2.00 = 0.25$.
- Midterm Practice 1.2.3 tutoring: $t^\star = 23/29 \approx 0.79$.
- Capstone airline rebooking: $t^\star = 100/360 \approx 0.28$.
- Capstone upsell email: $t^\star = 2/25 = 0.08$.

Every one is the same recipe, different payoff numbers.

### Connects to the exam
- This is **guaranteed material** on the midterm. Every version of the practice PDF has a decision-threshold short-answer.
- The formula $t^\star = C_{FP}/(B_{TP} + C_{FP})$ is the one you want on your cheat sheet.

---

# §8 · KNN — the non-parametric foil

## Step 1: What makes KNN different

Everything in §1–§7 has been **parametric**: pick a functional form ($y = \mathbf x' \beta + \varepsilon$, or $p = \sigma(\mathbf x'\beta)$), estimate the finite parameter vector $\beta$, make predictions.

KNN throws this entire machinery out. There is no $\beta$. There is no density assumption. There is no likelihood. The **whole training set is the model**.

For a new point $\mathbf x_{\text{new}}$:
1. Find its $K$ nearest neighbours in the training set, measured by some distance $d$.
2. Take their labels.
3. Predict by majority vote (classification) or average (regression).

**That's it. No training. No parameters. No optimization.**

## Step 2: Why KNN matters for this course

KNN exists in BUSN 20800 to illustrate a single point: **parametric assumptions are a choice, not a necessity**. Sometimes a completely local, assumption-free method works better.

| | parametric | non-parametric (KNN) |
|---|---|---|
| Model | fixed, finite $\beta$ | the data itself |
| Interpretability | coefficients = answers to \"what if\" | none |
| Scaling with $n$ | cheap prediction | $O(np)$ per query |
| Scaling with $p$ | regularization handles it | curse of dimensionality |
| Captures non-linearity | only if you hand-engineer features | automatically |

## Step 3: Distance — what \"nearest\" means

KNN needs a notion of closeness. Two distances matter:

- **Euclidean** $\sqrt{\sum (u_j - v_j)^2}$: for continuous features, after scaling.
- **Hamming** $\sum \mathbb 1\{u_j \neq v_j\}$: counts categorical disagreements. Use for 0/1 dummies.

**Why scale continuous features?** Euclidean distance is dominated by the largest-variance feature. \"Income\" (range 20k–200k) will swamp \"age\" (range 20–80) unless you standardize both to unit variance. Without scaling, KNN effectively ignores all but one feature.

**Why Hamming for dummies?** One-hot encoding creates 0/1 features where two households differing on exactly one categorical axis have Euclidean distance $\sqrt 2$ regardless of *which* axis. Hamming gives exactly 1 disagreement, which is the cleaner semantics.

Assignment 4 used Hamming on 9 household demographic dummies and beat multinomial logistic by ~15 pp — a textbook case of KNN winning because the feature space was purely categorical.

## Step 4: $K$ — the complexity knob

| $K$ | bias | variance | training accuracy | boundary |
|---|---|---|---|---|
| 1 | 0 | max | 100% (if rows unique) | very jagged |
| small | low | high | high | wiggly |
| moderate | moderate | moderate | moderate | smooth |
| $n$ | max (predicts modal class) | 0 | base rate | constant |

$K$ in KNN plays the role $\lambda$ plays in lasso — but **in the opposite direction**:

| knob | smaller = | bigger = |
|---|---|---|
| Lasso $\lambda$ | more flexible / overfit | more rigid / underfit |
| KNN $K$ | more flexible / overfit | more rigid / underfit |

Both are \"how much do I let the data drive the answer?\" Just with flipped conventions.

## Step 5: Training accuracy at $K = 1$ — why it's always 100%

If every training row has a distinct feature vector, its nearest neighbour is **itself** (distance 0). KNN at $K=1$ predicts every training point's own label. Training accuracy = 1.

This is the clearest signature of KNN's tendency to overfit. If test accuracy is nowhere near train accuracy, the $K$ is too small.

## Step 6: Why KNN boundaries are non-linear

The decision region for each class is the **union of Voronoi cells** (the set of points closer to training examples of that class than to any other). Voronoi cells can have arbitrary shape — curved, disconnected, whatever the data dictates.

This means KNN can carve boundaries logistic regression can't:
- Two positive clusters separated by a blob of negatives.
- Spirals, rings, other non-linearly-separable structures.

When the true decision boundary is non-linear and you have a lot of data, KNN will usually win. When the structure is linear or when $p$ is large, parametric methods dominate.

### Connects to the exam
- **Midterm Practice MC 4**: \"which method is non-parametric?\" → KNN.
- **Midterm Practice MC 6**: \"decreasing $K$ generally…\" → increases overfitting.
- **Midterm Practice 1.2.2**: compare KNN and logistic training accuracy → KNN higher because of non-linear flexibility.

---

# §9 · The map of methods — when to reach for which

Every method in this course sits somewhere in a 2D landscape. Two axes:

- **X axis: flexibility of the functional form** — is the model linear, GLM (link-wrapped linear), or fully non-parametric?
- **Y axis: capacity constraint** — no regularization (OLS/logistic), L1/L2 penalty (lasso/ridge), or natural capacity limit ($K$ in KNN)?

### The map

| | no regularization | L1 / L2 penalty | natural capacity limit |
|---|---|---|---|
| **Linear model** | OLS | **Lasso**, Ridge | (forward stepwise) |
| **GLM (link)** | logistic, probit | logistic + L1 (not covered) | |
| **Non-parametric** | | | **KNN** |

### Decision tree for method choice

```
Is y continuous?
├── yes → linear regression
│         ├── small p: plain OLS
│         ├── large p (p ≈ n or p > n): lasso (or ridge for non-sparse)
│         └── want unbiased coefficients in selected set: forward stepwise + OLS
│
├── no (binary y)?
│         ├── want interpretability (log-odds, odds ratios): logistic
│         ├── want Gaussian-latent-error interpretation: probit
│         └── want flexible non-linear boundary: KNN (with enough data)
│
├── no (K > 2 classes) → multinomial logistic (softmax)
│
└── completely unknown functional form + plenty of data + low p?
          → KNN (with standardization + CV for K)
```

## The course thesis, one more time

Everything in BUSN 20800 resolves the three crises of classical stats:
1. **Overfitting** in high dim → regularization (Chapter 5 / §5 of this notebook).
2. **Infeasible classical inference** → empirical CV and IC (Chapter 6 / §6).
3. **Probabilities instead of hypothesis tests** → decision thresholds (Chapter 7 / §7).

And KNN exists as a reminder that parametric assumptions are one choice among many.

---

# §10 · Reading the midterm — question archetypes

Midterm questions come in four shapes. Each shape wants a different mode of thinking.

## Archetype 1 — Multiple choice: *direction-of-effect*

These test your intuitions about how quantities change with a parameter:
- \"What happens to bias as $\lambda$ increases?\" — bias grows.
- \"As $K$ in KNN decreases, variance…\" — increases.
- \"Training $R^2 = 0.95$, test $R^2 = 0.30$: what's the concern?\" — overfitting.

**How to attack:** the answer is usually an arrow on a bias-variance or U-curve plot. Don't overthink — visualise the curve, read off the direction.

## Archetype 2 — Concept check: *mechanics*

These give you a small table or a formula and ask you to produce a number:
- Compute accuracy / sens / spec from a 2×2 matrix.
- Compute $R^2$ from given RSS and TSS.
- Determine how many fits $K$-fold requires.

**How to attack:** use the formulas from §11 of `MIDTERM_PREP.ipynb` or the \"six rates\" table in §7 above. Plug numbers. Show your work.

## Archetype 3 — Short answer: *derivation*

These ask you to reproduce a chain of algebraic moves:
- \"Derive the probit score function.\"
- \"Show that lasso is the MAP estimator under a Laplace prior.\"

**How to attack:** every derivation in this course is a **four-step recipe**:
1. Write down the density (Gaussian / Bernoulli / Laplace).
2. Take the iid product to get $L(\beta)$.
3. Take logs to get $\ell(\beta)$; apply chain rule / drop constants as needed.
4. Identify the result (OLS, lasso objective, score function).

If you can reproduce the recipe, you can regenerate the derivation from scratch. Don't memorise the final formula; memorise the *sequence of moves*.

## Archetype 4 — Short answer: *business decision*

These describe a scenario with costs and benefits and ask for an optimal action:
- Should the airline rebook this passenger?
- Should the university offer tutoring?
- At what threshold should the bank flag fraud?

**How to attack:** the five-step recipe from §7.6 above:
1. Write the payoff matrix.
2. Compute $\mathbb E[\text{act}\mid p]$ and $\mathbb E[\text{don't act}\mid p]$.
3. Set their difference to 0.
4. Solve linearly for $p$ — this is $t^\star$.
5. Report the direction of sensitivity / specificity going from 0.5 → $t^\star$.

Every decision problem reduces to filling in the payoff matrix and grinding through the algebra. **Nothing new** beyond the recipe.

---

## Your pre-exam mental rehearsal

Before the exam, practice saying the following out loud:

1. *\"The likelihood is the probability of the data as a function of the parameters.\"*
2. *\"Taking logs converts a product into a sum; it preserves the argmax.\"*
3. *\"OLS is the MLE under Gaussian errors.\"*
4. *\"A logistic coefficient is the change in log-odds, not probability.\"*
5. *\"The probit link is $\Phi$; its derivative is $\phi$.\"*
6. *\"Gradient descent exists because the logistic score equation has no closed form.\"*
7. *\"Lasso is the MAP with a Laplace prior; ridge is the MAP with a Gaussian prior.\"*
8. *\"Standardize before regularizing — otherwise the penalty is unit-sensitive.\"*
9. *\"$K$-fold CV with $n$ data and $K$ folds fits $K$ times on $n(K-1)/K$ rows.\"*
10. *\"The 1-SE rule picks the simplest model nearly tied with the CV minimum.\"*
11. *\"AIC and BIC fail on non-iid data because effective $n$ is smaller than raw $n$.\"*
12. *\"Sensitivity is TPR; specificity is TNR; precision is PPV.\"*
13. *\"Lowering the threshold raises sensitivity and lowers specificity.\"*
14. *\"The AUC is the probability a random positive ranks above a random negative.\"*
15. *\"The optimal threshold is $C_{FP} / (B_{TP} + C_{FP})$.\"*
16. *\"KNN majority-votes the K nearest training labels.\"*
17. *\"Decreasing K in KNN increases overfitting, in the same way that decreasing $\lambda$ in lasso does.\"*
18. *\"Euclidean for scaled continuous features; Hamming for categorical dummies.\"*

If you can say all 18 of these without hesitation, you are ready.

## Closing — what to remember in twenty years

Memory of specific formulas fades. What lasts is the intellectual skeleton:

- **Models are stories.** Pick a story; find the parameters that best explain the data.
- **Regularization is humility.** Let your beliefs about $\beta$ constrain what the data can say.
- **Evaluation is empirical.** Don't trust your training fit. Hold data out. Rehearse.
- **Decisions need payoffs.** A probability alone is not an action. Connect $\hat p$ to a cost matrix.
- **Parametric assumptions are a choice.** Sometimes the right answer is *not to assume*.

These five ideas are the permanent residue of BUSN 20800. Everything else is notation.